# Quiz answer adjudication

**Recipe 21** · Level 2 (Easy) · Decision type: `Choice`

Judge whether a typed quiz response matches an accepted answer by meaning, with partial-match and review outcomes for ambiguous responses.

Sources:

- S02: [TypeSafe AI: Primitives (Questions)](https://docs.typesafe.ai/primitives)
- S03: [TypeSafe AI: Confidence](https://docs.typesafe.ai/confidence)
- S04: [TypeSafe AI: How to build with TypeSafe](https://docs.typesafe.ai/concepts/how-to-build-with-system-one)
- S07: [TypeSafe AI: Jev 1.13 jaggedness](https://docs.typesafe.ai/model-jaggedness/jev-1.13)

## What you will build

A grader for short, typed quiz responses. Most responses never reach Jev at all: Python first
normalises the response (case, whitespace, punctuation, a leading article) and checks it against
a candidate set of accepted phrasings Python itself assembled; an exact match is graded `match`
on the spot. Only a response the normaliser cannot settle, such as a misspelling, an
over-specific answer, an unanticipated name, or a hedge between two options, is sent to Jev,
which picks one of four fixed outcomes: `match`, `partial_match`, `no_match`, or `needs_review`.
A response to a question whose accepted answer is a number is never sent to Jev at all, settled
or not: S07 says plainly that arithmetic belongs in code, so Python's own numeric comparison
grades every one of those by itself. By the end you will have seen both paths (settled and
model-assisted), the rule that decides when a model-chosen outcome becomes a final grade versus
a trip to a simulated human adjudicator, two trivial baselines the stored answers have to beat
to be worth calling Jev for at all, and an evaluation that reports how many responses never
needed a call, on top of the usual accuracy, per-class, and selective-prediction numbers.

## Setup and run mode

The notebook runs from its own folder with the standard library and `jev_cookbook` only.
`get_backend(fixtures=...)` replays the stored answers in `fixtures/responses.json` offline;
setting `JEV_COOKBOOK_LIVE=1` (see the README) swaps in live calls to the same question
definition and changes nothing else. `run_header` states which mode ran, and every number below
carries that mode with it. `fixtures/` holds 40 examples (19 `validation`, 19 `test`, 2 `demo`),
but `fixtures/responses.json` holds only 17 stored answers, one for every response the
normaliser cannot settle on its own: see "Python's part" and "Evaluation" below.

In [ ]:
import json

from jev_cookbook import get_backend, load_helpers
from jev_cookbook.evaluation import (
    accuracy,
    confusion_matrix,
    evaluate_outcomes,
    evaluate_selective,
    per_class_metrics,
    select_confidence_threshold,
    selective_curve,
)
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path, select_split
from jev_cookbook.simulation import ReviewQueue
from jev_cookbook.style import (
    apply_style,
    plot_answer_probabilities,
    plot_confusion_matrix,
    plot_risk_coverage,
    run_header,
    show_answer,
)

apply_style()
helpers = load_helpers()  # this recipe's helpers.py, loaded by path
examples = load_inputs()
labels = load_labels()
questions = helpers.build_questions()
backend = get_backend(fixtures=responses_path())
# N for the header is the number of examples the metrics are about: validation and test. The
# two demo examples are shown in this notebook but never scored.
scored = [e for e in examples if e.split != "demo"]

offline = backend.mode in ("synthetic", "scripted")
check = " (a pipeline check, not a Jev result)" if offline else ""
# CONTRIBUTING.md section 2: a number from the validation split is a selection step, not a
# result, in every run mode, not only an offline one. In a synthetic run a validation number
# carries BOTH labels, {selection}{check}: it is a selection step, and that selection step is
# itself only a pipeline check, not a Jev result.
selection = " (a selection step, not a reported result)"

# A response the normaliser settles never needs a request; one it cannot settle needs exactly
# one, cached here so a response shown individually and later scored in its split is decided
# once. The whole notebook's request count is therefore exactly the number of examples
# `helpers.settle` leaves unsettled, never more, whatever mode it runs in.
answered = {}


def get_answer(example):
    """Jev's answer for `example`, or `None` if `helpers.settle` already decided it without
    asking."""
    if helpers.settle(example.fields) is not None:
        return None
    if example.id not in answered:
        answered[example.id] = backend.decide(helpers.build_state(example.fields), questions)[
            "adjudication"
        ]
    return answered[example.id]


run_header(
    21,
    "Quiz answer adjudication",
    backend=backend,
    n_examples=len(scored),
)

## The state

The state is everything Jev sees, and Python builds it. Each example in `fixtures/inputs.jsonl`
carries `fields`: a `quiz_id` and the typed `response`. `quiz_id` is only a lookup key into
Python's own `QUESTION_BANK`; it never passes through the model. `build_state` looks the quiz up
and passes on three things: the `question` text, the `accepted_answers`, the candidate set of
phrasings Python is willing to call correct on sight, and the `response` itself. The candidate
set is assembled once, by Python, from `QUESTION_BANK`, the same data the normaliser checks
before any request is built, so what the reader sees here is exactly what the normaliser (and,
when it cannot settle things alone, Jev) is working from.

In [ ]:
demo = {e.id: e for e in examples if e.split == "demo"}
example = demo["d02-review"]
print("Python has:")
print(json.dumps(example.fields, indent=2))
state = helpers.build_state(example.fields)
print("Jev sees:")
print(json.dumps(state, indent=2))

## The questions

There is one question, asked only of a response the normaliser could not settle: does this
response match the accepted answer? A `Choice` fits because the use case names exactly four
outcomes Python is willing to act on, and Jev should commit to one of them rather than invent a
fifth. `needs_review` is not a last resort Python tacked on; it is one of the four options this
use case names explicitly, right alongside `match`, `partial_match` and `no_match`, because a
response that genuinely hedges between two options or gives no way to tell deserves a named
outcome of its own, not a forced guess. Each option's description says what belongs to it and,
for the pair that is easiest to confuse (`match` against `partial_match`), what moves a response
from one to the other: wording added before or after the accepted answer stays a `match` as
long as it does not substitute a different name; a response naming a role, a group, or someone
related to the accepted answer, but not the answer itself, is `partial_match` instead. Every
clause above is something Jev can check from `question`, `accepted_answers` and `response`
alone, nothing it would have to know independently (S07's point about a literal reading): an
earlier draft of the `match` criterion also allowed "extra detail that is itself true", which
Jev cannot check from the state at all, and that clause is gone. The instructions and every
option's description below are written to hold for any quiz question and any response this
recipe ever builds a state from: none of them names a specific accepted answer, a specific
response, or a specific quiz.

Two design choices keep Jev out of work Python already does better. First, the normaliser
(case, whitespace, punctuation and a leading article folded away) is checked before this
question is ever asked, so only a misspelling, an over-specific phrase, an unanticipated
alternative name, a hedge, or a genuinely wrong answer ever reaches it; TypeSafe's guidance on
building with System One (S04) is to "keep deterministic work in code" and let the model appear
"only where the system needs programmable common sense" (https://docs.typesafe.ai/concepts/how-to-build-with-system-one),
and an exact string match after normalising is as deterministic as it gets. Second, a question
whose accepted answer is a number is never sent here at all, settled or not: TypeSafe's notes
on Jev 1.13 (S07) say plainly that "Jev is not a calculator" and recommend implementing
mathematical logic in code (https://docs.typesafe.ai/model-jaggedness/jev-1.13), so Python's own
number parser, not a model judgment, decides every one of those.

A `Choice` answer's `confidence` is the top probability rescaled by the number of options,
`(p_max - 1/n) / (1 - 1/n)` (S03): 0 when the top option has no more than an even share of the
four, 1 when one option holds all the probability. "Python's part" below uses it as a gate on
top of whatever Jev picks.

In [ ]:
adjudication = questions["adjudication"]
print(adjudication.instructions)
for name, description in adjudication.criteria.items():
    print(f"  {name:14s} {description}")

## One answer up close

Before any aggregate, look at both paths a response can take. `d01-settled` normalises to an
exact match and never reaches this section's question at all. `d02-review` does not: it hedges
between two names, so it is sent to Jev, whose answer holds the chosen option, a probability for
every option, and the `confidence` explained above.

In [ ]:
settled_example = demo["d01-settled"]
print(f"response: {settled_example.fields['response']!r}")
print(f"normalised: {helpers.normalize_text(settled_example.fields['response'])!r}")
print(f"candidate set: {helpers.candidate_set(settled_example.fields['quiz_id'])}")
print(f"settled by the normaliser: {helpers.settle(settled_example.fields)}")
print(f"requests made for this example: {int(settled_example.id in answered)}")

In [ ]:
review_example = demo["d02-review"]
review_answer = get_answer(review_example)
show_answer(review_answer)
plot_answer_probabilities(review_answer, title="A hedging response, sent to Jev")

`d02-review`'s response, "Maybe Lindqvist, or possibly someone else entirely", names the
accepted answer but does not commit to it. The stored answer puts `needs_review` on top, which
is exactly the outcome a hedge like this should get under the policy "Python's part" states
next: a response that hedges between options is graded `needs_review`, never `no_match`, because
withholding a point from a response that may well be right is as much a side effect as awarding
one, and only a human adjudicator, not this notebook, should make that call.

## Python's part

`helpers.adjudicate` decides what happens with Jev's answer, and it checks the normaliser
first: when `helpers.settle` can decide a response alone, Jev's answer is never read, not even
if one happens to exist. Once a response does reach Jev, two independent conditions each send it
to `needs_review` instead of reporting a grade: Jev choosing `needs_review` itself, which this
rule never second-guesses regardless of confidence, and a confidence below the confidence gate
for any of the other three choices. `needs_review` is never itself a final score. CONTRIBUTING.md
section 4 treats awarding or withholding a point as a side effect that must stay in Python's
control, so every response that lands on `needs_review`, whichever of the two conditions put it
there, goes into a simulated `ReviewQueue` for a human adjudicator to resolve, not into a grade
this rule reports on its own. This is the fallback-option rule from
[docs/recipe-template.md](../../docs/recipe-template.md): `needs_review` could, in principle,
stand as a final result on its own if choosing it had no side effect, the way the template's
`none` option can; it cannot here, because withholding or deferring a grade is exactly the kind
of effect the rule in CONTRIBUTING.md section 4 is written to keep in Python's hands.

A hedge between two options is one of this recipe's hard cases, and it needs a stated policy,
applied the same way everywhere: this recipe's fixtures always give a hedging response the
**gold label** (the outcome a human adjudicator would have assigned in advance, used only to
score the rule afterwards, never shown to Jev) `needs_review`, never `no_match`, because a
response that names the right answer alongside a wrong one is not the same claim as a response
that is simply wrong, and collapsing the two would teach the rule to penalise honesty about
uncertainty.

The one setting this recipe chooses is a **confidence gate**: a cut-off on `confidence` below
which an answer is sent to `needs_review` instead of being graded. It is chosen here, on
`validation`, from only the responses Jev was actually asked about (the normaliser's own
matches have no confidence to gate): `select_confidence_threshold` picks the lowest confidence
gate whose accepted subset of those responses is perfectly accurate on `validation`, the most
such answers can be kept automatic without accepting one that is *wrong* on `validation` itself.
The validation fixtures include one response, `v12-reef-misspell`, whose stored answer is
confidently wrong in the other direction (leaning `no_match` on what the gold label calls a
recognisable misspelling), so this selection has a real wrong answer to set the bar against, not
a vacuous one. Two of the eight responses that feed this selection, `v04-capital-hedge` and
`v07-guild-hedge`, are ones the gate can never actually govern: Jev's own raw choice for both is
already `needs_review`, the same as their gold label, so `adjudicate`'s unconditional branch
answers for them before any confidence is read. Keeping them in the selector's input is
harmless here, since a response that is always "correct" cannot force the gate higher the way a
genuine wrong answer would, but a recipe whose fixtures made that answer wrong at a high
confidence would need to leave such responses out instead.

The frozen gate is then demonstrated on both examples shown above, adding the review queue's
first item; "Evaluation" below fills the same queue with every other response this notebook
ever sends to review.

In [ ]:
val_examples = select_split(examples, "validation")
val_called = [e for e in val_examples if helpers.settle(e.fields) is None]
val_called_answers = [get_answer(e) for e in val_called]
val_called_gold = [labels[e.id] for e in val_called]
# v04-capital-hedge and v07-guild-hedge are always "correct" here (Jev's raw choice is
# needs_review, which is also their gold label), so they are in this array even though the
# gate below can never actually govern either one -- see "Python's part" above.
val_called_correct = [
    a.choice == g for a, g in zip(val_called_answers, val_called_gold, strict=True)
]
val_called_confidence = [a.confidence for a in val_called_answers]
threshold = select_confidence_threshold(
    val_called_correct, val_called_confidence, target_accuracy=1.0
)
print(
    f"validation responses sent to Jev: {len(val_called)} of {len(val_examples)}{selection}{check}"
)
print(f"chosen confidence gate, frozen from here on: {threshold:.4f}{selection}{check}")

queue = ReviewQueue()
for ex in (settled_example, review_example):
    ans = get_answer(ex)
    result = helpers.adjudicate(ex.id, ex.fields, ans, threshold)
    print(f"{result.example_id}: {result.outcome} ({result.reason})")
    if result.outcome == helpers.NEEDS_REVIEW:
        queue.submit(helpers.queue_item(ex.fields), result.reason, answer=ans)
print(f"review queue so far: {len(queue)} item(s){selection}{check}")

## Evaluation

The confidence gate was chosen and frozen above, on `validation`. This section applies
`helpers.adjudicate` to every example in `validation` and in `test` (`test` once, to report),
and every count below comes from what it actually returns, not from a separate reimplementation
of the same rule. It covers, in order: how many of each split's responses the normaliser settled
without a call, agreement between the rule's outcome and the human gold label, a confusion
matrix and
per-class precision and recall on `test`, the `needs_review` outcome broken down by which of the
two conditions sent it there, two trivial baselines the responses sent to Jev have to beat,
selective coverage, accuracy and risk for the confidence gate among only the responses actually
sent to Jev, and finally a direct, numeric comparison of `evaluate_outcomes` against
`evaluate_selective` on the same responses. In an offline run every `test` number here is a
check that the pipeline works, and it says so; every `validation` number, in every run mode, is
a selection step rather than a reported result.

In [ ]:
def adjudicate_split(split_examples, min_confidence):
    """Apply the rule to every example in a split and tally what it actually returned."""
    return [
        helpers.adjudicate(e.id, e.fields, get_answer(e), min_confidence) for e in split_examples
    ]


def called_only(split_examples, results):
    """``(example, result)`` pairs for the responses that were actually sent to Jev."""
    return [
        (e, r) for e, r in zip(split_examples, results, strict=True) if not r.settled_without_a_call
    ]


test_examples = select_split(examples, "test")
val_results = adjudicate_split(val_examples, threshold)
test_results = adjudicate_split(test_examples, threshold)

n_settled_val = sum(r.settled_without_a_call for r in val_results)
n_settled_test = sum(r.settled_without_a_call for r in test_results)
print(
    f"validation: {n_settled_val} of {len(val_examples)} responses settled without a model "
    f"call{selection}{check}"
)
print(
    f"test: {n_settled_test} of {len(test_examples)} responses settled without a model call{check}"
)
print(
    f"requests made so far in this notebook: {len(answered)} "
    f"(every settled response above cost none){check}"
)

That second metric, how many responses never needed a model call, is reported as its own
number here, not folded into anything else: it comes from `r.settled_without_a_call`,
the same field `helpers.adjudicate` sets when `helpers.settle` decides a response alone, so this
count can never silently drift from what the rule actually did.

In [ ]:
val_gold = [labels[e.id] for e in val_examples]
test_gold = [labels[e.id] for e in test_examples]
val_outcome = [r.outcome for r in val_results]
test_outcome = [r.outcome for r in test_results]

print(
    f"validation agreement with the gold label: {accuracy(val_gold, val_outcome):.4f}{selection}{check}"
)
print(f"test agreement with the gold label: {accuracy(test_gold, test_outcome):.4f}{check}")

"Agreement" counts a response as right whenever the rule's final outcome exactly equals the
gold label, `needs_review` included: a hedge correctly routed to `needs_review` agrees with a
gold label of `needs_review` exactly as a confident `match` agrees with a gold label of `match`.
It can also read as stricter than it sounds in one direction: a response the confidence gate
routes to `needs_review` because Jev's own top choice was *not* `needs_review` (`v12-reef-misspell`
on `validation` is exactly this) counts as disagreement here, even though routing it to a human
rather than reporting Jev's wrong top choice as a grade is the safer of the two mistakes the rule
could make. The counts right after the confusion matrix below separate `needs_review` out on its
own, which is where that distinction is visible.

In [ ]:
matrix = confusion_matrix(test_gold, test_outcome, labels=list(helpers.OUTCOMES))
print(f"gold (row) by predicted (column), {len(helpers.OUTCOMES)} outcomes{check}:")
for row_label, row in zip(matrix.labels, matrix.matrix, strict=True):
    print(f"  {row_label:14s} {list(int(v) for v in row)}")
plot_confusion_matrix(matrix, title=f"Test split, {len(test_examples)} examples{check}")

For each outcome, **precision** is the fraction of the rule's calls of that outcome that were
actually right, and **recall** is the fraction of that outcome's real `test` responses the rule
found at all; **f1** is their harmonic mean, a single number that is low whenever either one is,
and **support** is how many `test` responses actually carry that gold label, so a recall or
precision figure is only as meaningful as its support is large.

In [ ]:
per_class = per_class_metrics(test_gold, test_outcome, labels=list(helpers.OUTCOMES))
for label, counts in per_class.items():
    print(
        f"{label:14s} precision {counts.precision:.3f}  recall {counts.recall:.3f}  "
        f"f1 {counts.f1:.3f}  support {counts.support}{check}"
    )

`no_match`'s `test` support of 6 is three responses the normaliser itself settled to
`no_match` (`t15`, `t17` and `t19`, all number questions) plus three sent to Jev (`t02`, `t04`
and `t05`), so its precision and recall describe more than a single example either way.
`needs_review`'s only `test` fixture with that gold label is `t12`, reached by Jev's own choice
rather than the confidence gate; `t10`'s gold label is `partial_match`, and it is the confidence
gate, not Jev's own choice, that routes it to `needs_review` instead, which is why the breakdown
right after this cell separates the two causes of a `needs_review` outcome rather than leaving
them pooled.

In [ ]:
def review_breakdown(results):
    reasons = {}
    for r in results:
        if r.outcome == helpers.NEEDS_REVIEW:
            reasons[r.reason] = reasons.get(r.reason, 0) + 1
    return reasons


for split_name, results, n, lbl in (
    ("validation", val_results, len(val_examples), f"{selection}{check}"),
    ("test", test_results, len(test_examples), check),
):
    reasons = review_breakdown(results)
    print(f"{split_name}: sent to needs_review: {sum(reasons.values())} of {n}{lbl}")
    for reason, count in sorted(reasons.items()):
        print(f"  because {reason}: {count}{lbl}")

### Filling the review queue

The counts above are a tally; `queue`, created in "Python's part" with one `demo` item in it
already, is the actual simulated side effect a `needs_review` outcome triggers. The cell below
adds every `validation` and `test` response whose outcome is `needs_review` -- not only the one
`demo` example shown earlier -- each carrying the quiz id, the question, and the response itself
(`helpers.queue_item`), so a human adjudicator opening the queue has something to read, then
prints the whole thing.

In [ ]:
for split_name, split_examples, results, lbl in (
    ("validation", val_examples, val_results, f"{selection}{check}"),
    ("test", test_examples, test_results, check),
):
    added = 0
    for e, r in zip(split_examples, results, strict=True):
        if r.outcome == helpers.NEEDS_REVIEW:
            queue.submit(helpers.queue_item(e.fields), r.reason, answer=get_answer(e))
            added += 1
    print(f"{split_name}: {added} item(s) added to the queue just now{lbl}")

print(f"review queue, {len(queue)} item(s) total, oldest first{check}:")
for row in queue.to_dicts():
    item = row["item"]
    print(f"  {item['quiz_id']:8s} {item['response']!r} -- {row['reason']}")

### Two trivial baselines

Calling Jev is only worth it if it beats something much cheaper. Both baselines below are
chosen on `validation`'s **called** responses only, the ones the normaliser could not settle on
its own: a response the normaliser already settles is not a judgment Jev, or any baseline,
makes, so including those would flatter any baseline equally and compare nothing.

The **majority-class baseline** always predicts whichever of the four outcomes is most common
among `validation`'s called responses. The **string-similarity baseline** compares the
normalised response against the normalised candidate set with Python's own
`difflib.SequenceMatcher` ratio and predicts `match` above a cut-off chosen on the same
`validation` responses, `no_match` otherwise; it can never predict `partial_match` or
`needs_review` at all. That is not a tuning gap, it is the baseline's ceiling: three of
`test`'s eight called responses (`t07`, `t10`, `t12`) have one of those two gold labels, so a
baseline that can only say `match` or `no_match` is wrong on every one of them, by
construction, however its cut-off is chosen.

In [ ]:
import difflib


def majority_label(golds):
    counts = {}
    for g in golds:
        counts[g] = counts.get(g, 0) + 1
    return max(counts, key=counts.get)


def best_similarity(fields):
    quiz = helpers.QUESTION_BANK[fields["quiz_id"]]
    response_norm = helpers.normalize_text(fields["response"])
    return max(
        difflib.SequenceMatcher(None, response_norm, helpers.normalize_text(a)).ratio()
        for a in quiz["accepted"]
    )


val_called_pairs = called_only(val_examples, val_results)
test_called_pairs = called_only(test_examples, test_results)
val_called_gold_list = [labels[e.id] for e, _ in val_called_pairs]
test_called_gold_list = [labels[e.id] for e, _ in test_called_pairs]

majority = majority_label(val_called_gold_list)
print(f"majority gold label among validation's called responses: {majority!r}{selection}{check}")


def sim_accuracy(t):
    preds = ["match" if best_similarity(e.fields) >= t else "no_match" for e, _ in val_called_pairs]
    return accuracy(val_called_gold_list, preds)


candidate_thresholds = sorted({round(best_similarity(e.fields), 6) for e, _ in val_called_pairs})
sim_threshold = max(candidate_thresholds, key=sim_accuracy)
print(f"chosen similarity threshold, frozen from here on: {sim_threshold:.4f}{selection}{check}")

majority_preds = [majority] * len(test_called_pairs)
sim_preds = [
    "match" if best_similarity(e.fields) >= sim_threshold else "no_match"
    for e, _ in test_called_pairs
]
rule_preds = [r.outcome for _, r in test_called_pairs]

print(
    f"test, responses sent to Jev only ({len(test_called_pairs)} of {len(test_examples)}){check}:"
)
print(
    f"  majority-class baseline accuracy: {accuracy(test_called_gold_list, majority_preds):.4f}{check}"
)
print(
    f"  string-similarity baseline accuracy: {accuracy(test_called_gold_list, sim_preds):.4f}{check}"
)
print(
    f"  this recipe's rule, agreement with gold: {accuracy(test_called_gold_list, rule_preds):.4f}{check}"
)

The stored answers clearly beat both baselines on the responses that were actually sent to Jev,
by more than the one or two fixtures either baseline gets right or wrong by chance: the
majority-class baseline is capped by how lopsided `test`'s called gold labels are, and the
similarity baseline is capped at five of eight whatever cut-off it picks, because `t07`, `t10`
and `t12` are not expressible as `match` or `no_match` at all. This margin is true by
construction, not a result either baseline or Jev produced: every stored probability in `ROWS`
was written by hand, keyed on the gold label, so it shows the two baselines' structural
ceilings against fixtures authored to clear them, not a capability anything demonstrated live.
If a later edit to `ROWS` ever closes that gap, it is telling you the fixtures stopped giving
Jev anything to add over a string comparison, not that the rule above is broken.

The confidence gate only ever matters for responses that were actually sent to Jev; a response
the normaliser settled has no confidence to gate and is accepted with certainty. **Coverage** is
the fraction of those called responses the rule actually answers, as a grade, rather than
sending to `needs_review`; among those answered, **accuracy** is the fraction that are right,
and **risk** is its complement, `1 - accuracy`, the fraction of answered responses that are
wrong despite having been graded. Because this rule's `needs_review` has an unconditional branch
of its own, Jev choosing it outright on top of the confidence gate, `evaluate_selective` cannot
be used for these numbers: it would reapply only the confidence comparison and silently disagree
with what the rule actually did whenever Jev's own choice is the reason. `evaluate_outcomes`
instead takes the rule's own accept/review decisions directly; "Why `evaluate_outcomes`" below
makes the disagreement concrete rather than only arguing it.

In [ ]:
def selective_report(pairs, gold, label, suffix):
    accepted = [r.outcome != helpers.NEEDS_REVIEW for _, r in pairs]
    correct = [r.outcome == gold[e.id] for e, r in pairs]
    result = evaluate_outcomes(accepted, correct)
    print(
        f"{label}: answered {result.n_answered} of {result.n_total} responses sent to Jev "
        f"(coverage {result.coverage:.4f}){suffix}"
    )
    print(f"{label}: accuracy among those answered: {result.accuracy:.4f}{suffix}")
    print(f"{label}: risk among those answered: {result.risk:.4f}{suffix}")
    return result


selective_val = selective_report(
    val_called_pairs, labels, "validation, called subset", f"{selection}{check}"
)
selective_test = selective_report(test_called_pairs, labels, "test, called subset", check)

`test`'s risk above is not zero, on purpose: `t04-capital-wrong-confident` is answered `match`
at a confidence the validation-frozen confidence gate does not catch, when the gold label is
`no_match`. Its response, "Caldenhall", names a different place entirely from the accepted
answer, "Mirrowgate"; the stored answer is deliberately overconfident about it, the way a model
that leans on superficial plausibility between two similarly grand-sounding fictional place
names might. A confidence gate chosen to be perfectly accurate on `validation` is a guarantee about
`validation`, not about `test` or any response this recipe has not seen, and this fixture is why
that gap is worth keeping, not averaging away.

### Why `evaluate_outcomes`

`t12-reef-hedge`'s stored answer is confidently `needs_review`, 0.80, well above the frozen
gate. `evaluate_selective` reapplies the gate to every raw confidence as if it were the rule's
only review branch, so it would count `t12` as "answered" (confidence above the gate) and, by
Jev's own raw choice, "correct" (`needs_review` is also the gold label). `helpers.adjudicate`
never checks `t12`'s confidence at all: Jev chose `needs_review` outright, so the unconditional
branch sends it to review regardless, and it is never counted as answered. The cell below
computes both views on the same responses and prints them side by side, on `validation` and on
`test`, rather than only asserting that they differ.

In [ ]:
def naive_selective(pairs, gold):
    correct = [get_answer(e).choice == gold[e.id] for e, _ in pairs]
    confidence = [get_answer(e).confidence for e, _ in pairs]
    return evaluate_selective(correct, confidence, threshold)


naive_val = naive_selective(val_called_pairs, labels)
naive_test = naive_selective(test_called_pairs, labels)

for label, outcomes_result, selective_result, suffix in (
    ("validation", selective_val, naive_val, f"{selection}{check}"),
    ("test", selective_test, naive_test, check),
):
    print(
        f"{label}, evaluate_outcomes: coverage {outcomes_result.coverage:.4f}, "
        f"accuracy {outcomes_result.accuracy:.4f}, risk {outcomes_result.risk:.4f}{suffix}"
    )
    print(
        f"{label}, evaluate_selective: coverage {selective_result.coverage:.4f}, "
        f"accuracy {selective_result.accuracy:.4f}, risk {selective_result.risk:.4f}{suffix}"
    )
    agree = (
        outcomes_result.coverage == selective_result.coverage
        and outcomes_result.accuracy == selective_result.accuracy
    )
    print(f"{label}: the two {'coincide' if agree else 'disagree'}{suffix}")

On `validation` the two coincide: no called response there has Jev confidently choosing
`needs_review` above the gate, so the unconditional branch never has a chance to diverge from a
plain confidence comparison on this split, and the printed line says so rather than assuming it.
On `test`, `t12-reef-hedge` makes them disagree exactly as described above:
`evaluate_selective` reports a larger coverage (it counts `t12` as answered) and, because `t12`
happens to be right by Jev's raw choice, the same or a smaller risk than `evaluate_outcomes`,
which correctly reports `t12` as reviewed, not answered. `evaluate_selective`'s numbers describe
a rule this notebook does not run, one where `needs_review` is gated on confidence like every
other outcome; `evaluate_outcomes`, fed what `helpers.adjudicate` actually decided, is the one
that is true of it.

In [ ]:
# Confidence-only: selective_curve knows nothing about adjudicate's unconditional needs_review
# branch, so it reapplies `confidence >= t` to Jev's raw choice, the same simplification
# evaluate_selective makes (see "eval-curve-md" below for what that does and does not show).
val_called_choice_correct = [get_answer(e).choice == labels[e.id] for e, _ in val_called_pairs]
val_called_conf_for_curve = [get_answer(e).confidence for e, _ in val_called_pairs]
curve = selective_curve(val_called_choice_correct, val_called_conf_for_curve)
print("validation sweep (called subset), confidence-only view, coverage -> risk, rounded:")
for t, cov, risk in zip(curve.thresholds, curve.coverage, curve.risk, strict=True):
    print(f"  threshold {t:.4f}: coverage {cov:.4f}  risk {risk:.4f}{selection}{check}")
plot_risk_coverage(
    curve.coverage,
    curve.risk,
    label="validation, called subset (confidence-only)",
    title=f"Confidence-only sweep, validation called subset{check}",
)

This is the **confidence-only** view of the same called subset, not what `helpers.adjudicate`
itself would do at a looser gate: `selective_curve` only ever reapplies `confidence >= t`, so
its two lowest rows (thresholds 0.4000 and 0.3333) credit `v04-capital-hedge` and
`v07-guild-hedge` as answered once the cut-off drops low enough to include their confidence.
The rule itself never answers either one, at this gate or any looser one: Jev's own choice for
both is `needs_review` outright, which `adjudicate`'s unconditional branch never lets become a
grade, so this rule's own coverage on this called subset cannot exceed 6 of 8 (0.7500), the
frozen gate's own coverage reported above, however far the gate is loosened. No outcome-based
sweep helper exists yet in `jev_cookbook.evaluation` (see "Selective prediction" in
docs/evaluation.md and issue #161 for the request); the curve above is the closest available
view, and is read here as confidence-only, consistent with "Why `evaluate_outcomes`" rather than
contradicting it.

## What was and was not measured

What this run can say about Jev depends on its mode, which the cell below states from
`backend.mode`: a synthetic or scripted run is a pipeline check only, on the fixture sample it
ran against; a recorded or live run states the model the API returned and when, for exactly the
examples it covers. Nothing here says anything about Jev's quality, speed or cost unless that
cell reports a recorded or live run. `N` below counts only the examples that carry a gold label,
so it excludes the two `demo` examples shown earlier.

In [ ]:
if offline:
    print(f"Provenance: {backend.mode}. Not measured live: a pipeline check, not a Jev result.")
else:
    dates = ", ".join(getattr(backend, "recorded_dates", ()) or ()) or "this run"
    print(f"Provenance: {backend.mode}, model {backend.model}")
    print(f"Captured: {dates}")
    print(f"N: {len(val_examples)} validation and {len(test_examples)} test examples")

## Next steps

- Add a hard response to `ROWS` in `build_fixtures.py` with the answer you expect a model might
  give, then run `python build_fixtures.py`. If you give it a `quiz_id` and `response` whose
  normalised form does not match any row already in the fixtures, expect
  `SystemExit: ROWS disagrees with helpers.settle` the first time -- that check is there on
  purpose, to catch a `spec` that no longer agrees with what the normaliser would do; add or fix
  the row until it passes, then see where `helpers.adjudicate` sends it.
- Change the hedge policy: make a hedging response's gold label `no_match` instead of
  `needs_review` in a couple of `test` rows (`t12-reef-hedge` is one), **then re-run
  `python build_fixtures.py`** -- editing `labels.jsonl`'s source in `ROWS` alone changes
  nothing until the fixtures are regenerated -- and see how the per-class numbers for both
  outcomes move.
- Neighbouring recipes: [`01-sentiment-classification`](../01-sentiment-classification/) (a
  `Choice` over a fixed option set like this one), and
  [`36-card-game-action-selection`](../36-card-game-action-selection/), another
  `Games & simulation` recipe built the same way: Python enumerates the fixed options and Jev
  only ever picks among them.